# 06 — MCMC Fitting a Stacked Spectrum

This notebook demonstrates MCMC fitting on a rest-frame stacked spectrum
where the resolving power must be estimated from pixel spacing.

It covers:
1. Loading a `.npz` stacked spectrum and estimating R(λ)
2. MCMC fit with `emcee` — BIC broad selection is on by default
3. Per-line results with asymmetric credible intervals
4. Convergence diagnostics (R-hat, ESS)
5. Trace plots, flux posteriors, and corner plots
6. Flux-ratio posteriors (O3/Hβ) for metallicity diagnostics
7. Compare MCMC posteriors with bootstrap uncertainties from `jwspecfit`
8. Plot the MCMC fit using `jwspecfit.plot_fit()`
9. Narrow-only MCMC for speed
10. Custom priors

In [ ]:
import jwspecfit
import jwspecmcmc
import matplotlib.pyplot as plt
import numpy as np

print(f"jwspecfit  v{jwspecfit.__version__}")
print(f"jwspecmcmc v{jwspecmcmc.__version__}")

## Load the stacked spectrum

This is a rest-frame UV–optical stack of 129 galaxies at z ~ 6, with
wavelengths in Angstroms and flux in arbitrary units.

R(λ) is estimated from the pixel spacing using `R_from_pixels`,
since stacked spectra have no associated grating.

In [ ]:
stack = jwspecfit.read_npz(
    "../../data/stack_all_Muv19_21_DustCorrected.npz",
    z=0.0,     # already in rest frame
)

# Estimate R from pixel spacing (no grating info for stacks)
R_fn = jwspecfit.R_from_pixels(stack.wave_um)
stack.R = R_fn

print(f"{stack.n_pix} pixels")
print(f"Wave: {stack.wave_A.min():.0f} – {stack.wave_A.max():.0f} Å")
print(f"N stacked: {stack.meta.get('n_stacked', '?')}")

## Plot R(λ)

Show the estimated resolving power as a function of wavelength.

In [ ]:
valid = stack.mask_valid()
R_est = R_fn(stack.wave_um)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(stack.wave_A[valid], R_est[valid], lw=0.8)
ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Estimated R")
ax.set_title("Resolving power estimated from pixel spacing")
plt.tight_layout()

## Quick-look plot

In [ ]:
valid = stack.mask_valid()

fig, ax = plt.subplots(figsize=(12, 3))
ax.step(stack.wave_A[valid], stack.flux_ujy[valid], where="mid", lw=0.5, color="0.3")
ax.set_xlabel(r"Rest wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Flux")
ax.set_title("Stacked UV spectrum (129 galaxies)")
plt.tight_layout()

## MCMC fit with emcee

A single call mirrors `jwspecfit.fit_lines()` but uses MCMC sampling.
By default (`mode="auto"`), BIC model selection checks for broad Balmer
components before running MCMC on the winning model.

For stacked spectra we pass `z=0.0` (rest frame) and specify the
optical lines of interest.

In [ ]:
optical_lines = ["HBETA", "OIII_5007", "Ha", "NII_6549", "NII_6585"]

result = jwspecmcmc.fit_lines(
    stack, z=0.0,
    lines=optical_lines,
    sampler="emcee",
    n_steps=2000,
    n_boot_bic=0,  # single-point BIC for speed (set >0 for bootstrap BIC)
    seed=42,
)

print(f"Selected:    {result.selected_model}")
print(f"Sampler:     {result.sampler_name}")
print(f"Lines:       {', '.join(result.lines.keys())}")
print(f"Chain shape: {result.flat_chains.shape}")
print(f"Burn-in:     {result.sampler_meta.get('n_burn', '?')} steps")
print(f"BIC narrow:  {result.bic_narrow:.1f}")
print(f"BIC broad1:  {result.bic_broad1:.1f}")

## Per-line results with asymmetric uncertainties

MCMC gives proper (16th, 84th) percentile credible intervals,
which can be asymmetric for parameters near bounds.  Broad
components are flagged with `*`.

In [ ]:
print(f"{'Line':<22s} {'Flux':>12s} {'−err':>12s} {'+err':>12s} {'SNR':>8s} {'EW (Å)':>10s}")
print("-" * 80)
for name, lr in result.lines.items():
    tag = " *" if "BROAD" in name else ""
    print(
        f"{name + tag:<22s} {lr.flux:12.3e} {lr.flux_err[0]:12.3e} {lr.flux_err[1]:12.3e}"
        f" {lr.snr:8.1f} {lr.ew_A:10.1f}"
    )

## Convergence diagnostics

The Gelman–Rubin R-hat should be < 1.05 and the effective sample
size (ESS) should be > 100 for all parameters.

In [ ]:
conv = result.convergence
print(f"R-hat max:  {conv.get('r_hat_max', 'N/A'):.3f}")
print(f"ESS min:    {conv.get('ess_min', 'N/A'):.0f}")
print(f"Converged:  {conv.get('converged', 'N/A')}")

if 'r_hat' in conv:
    r_hat = conv['r_hat']
    print(f"\nR-hat range: {np.nanmin(r_hat):.4f} – {np.nanmax(r_hat):.4f}")
    n_bad = np.sum(r_hat > 1.05)
    if n_bad > 0:
        print(f"WARNING: {n_bad} parameters have R-hat > 1.05")

## Trace plots

Inspect how the chains explore parameter space.  Good chains
should look like "hairy caterpillars" with no trends.

In [ ]:
fig = jwspecmcmc.plot_traces(
    result,
    params=["A_OIII_5007", "A_Ha", "A_HBETA", "sigma_OIII_5007"],
)
plt.show()

## Flux posterior histograms

Posterior distributions for the brightest lines, with
median and 68% credible interval.

In [ ]:
bright_lines = [name for name, lr in result.lines.items() if lr.snr > 5]
n_bright = len(bright_lines)

if n_bright > 0:
    ncols = min(n_bright, 3)
    nrows = (n_bright + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, line in zip(axes, bright_lines):
        jwspecmcmc.plot_flux_posterior(result, line, ax=ax)
    for ax in axes[n_bright:]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No lines with SNR > 5.")

## Corner plot

Joint posterior of Hα, Hβ, and [OIII] 5007 amplitudes.
Off-diagonal panels show covariances between parameters.

In [ ]:
fig = jwspecmcmc.plot_corner(
    result,
    params=["A_Ha", "A_HBETA", "A_OIII_5007"],
)
plt.show()

## Flux-ratio posteriors

Compute the posterior on O3/Hβ = [OIII] 5007 / Hβ,
which is a key diagnostic for metallicity and ionisation.

In [ ]:
if "OIII_5007" in result.lines and "HBETA" in result.lines:
    ratio = result.flux_ratio_posterior("OIII_5007", "HBETA")
    ratio_clean = ratio[np.isfinite(ratio) & (ratio > 0)]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(ratio_clean, bins=50, density=True, alpha=0.7, color="C2", edgecolor="C2")
    med = np.median(ratio_clean)
    lo, hi = np.percentile(ratio_clean, [16, 84])
    ax.axvline(med, color="C1", ls="-", lw=1.5, label=f"Median = {med:.2f}")
    ax.axvline(lo, color="C1", ls="--", lw=1.0, label=f"16th = {lo:.2f}")
    ax.axvline(hi, color="C1", ls="--", lw=1.0, label=f"84th = {hi:.2f}")
    ax.set_xlabel(r"[OIII] 5007 / H$\beta$")
    ax.set_ylabel("Probability density")
    ax.legend(fontsize=9)
    ax.set_title(r"O3/H$\beta$ flux ratio posterior")
    plt.tight_layout()
    plt.show()
else:
    print("OIII_5007 or HBETA not in fitted lines.")

## Compare with bootstrap (LSQ)

Run the standard `jwspecfit.fit_with_broad()` with bootstrap
uncertainties and compare error bars with the MCMC result.

In [ ]:
result_lsq = jwspecfit.fit_with_broad(
    stack, z=0.0,
    lines=optical_lines,
    mode="auto",
)

print(f"{'Line':<22s} {'Flux (LSQ)':>12s} {'Boot err':>12s} {'MCMC −err':>12s} {'MCMC +err':>12s}")
print("-" * 74)
for name in result_lsq.best_fit.lines:
    if name in result.lines:
        lr_lsq = result_lsq.best_fit.lines[name]
        lr_mc = result.lines[name]
        print(
            f"{name:<22s} {lr_lsq.flux:12.3e} {lr_lsq.flux_err:12.3e}"
            f" {lr_mc.flux_err[0]:12.3e} {lr_mc.flux_err[1]:12.3e}"
        )

## Plot the MCMC fit

Convert to `FitResult` and use `plot_fit()` for the standard
data + model + residuals view.  For stacked spectra, plot in
Angstroms.

In [ ]:
fit_result = result.to_fit_result()
print(f"χ²/dof = {fit_result.chi2:.2f}")

fig = jwspecfit.plot_fit(fit_result, wave_unit="A", flux_unit="flam")
fig.suptitle("MCMC median posterior fit (stacked spectrum)", y=1.02)
plt.show()

In [ ]:
fig_interactive = jwspecfit.plot_fit_interactive(fit_result, wave_unit="A")
fig_interactive.show()

## Narrow-only MCMC

Use `mode="off"` to skip broad-component selection and fit only
narrow lines.  This is faster and useful when broad components
are not expected.

In [ ]:
result_narrow = jwspecmcmc.fit_lines(
    stack, z=0.0,
    lines=optical_lines,
    mode="off",
    n_steps=1000,
    seed=42,
)

print(f"Selected:    {result_narrow.selected_model}")
print(f"Lines:       {', '.join(result_narrow.lines.keys())}")
print()
for name, lr in result_narrow.lines.items():
    print(f"  {name:<18s} flux={lr.flux:.3e} (-{lr.flux_err[0]:.3e}, +{lr.flux_err[1]:.3e})  SNR={lr.snr:.1f}")

## Custom priors

Override the default uniform priors with a Gaussian prior on a specific
parameter.  Useful when you have prior knowledge from other observations
or theoretical expectations.

In [ ]:
from jwspecmcmc import GaussianPrior

# Example: informative Gaussian prior on Ha amplitude
result_prior = jwspecmcmc.fit_lines(
    stack, z=0.0,
    lines=optical_lines,
    mode="off",
    n_steps=1000,
    prior_overrides={
        "A_Ha": GaussianPrior(mean=3e-16, std=5e-17, lo=0, hi=1e-14),
    },
)

print("With Gaussian prior on A_Ha:")
for name, lr in result_prior.lines.items():
    print(f"  {name:<18s} flux={lr.flux:.3e} (-{lr.flux_err[0]:.3e}, +{lr.flux_err[1]:.3e})")